# PCA Example — From Scratch

This notebook walks through **Principal Component Analysis (PCA)** with the
custom implementation in `rice_ml.unsupervised_learning.pca`.

PCA is an **unsupervised dimensionality reduction method**. Its goal is to
replace a high-dimensional feature space with a smaller set of directions that
retain as much of the original variation as possible.

In this notebook, we will:

- load the Wine Quality dataset from a public source
- explore the data before applying PCA
- standardize the features
- fit PCA step by step
- examine explained variance and scree plots
- visualize the data in two principal components
- summarize what PCA helps us learn from the dataset


## Mathematical Intuition

Given a dataset represented by a feature matrix:

- Feature matrix:  
  `X ∈ ℝ^{n × d}`

PCA seeks a set of **orthonormal directions**  
`{v₁, v₂, …, v_k}` such that the projected data has **maximum variance**.

---

### Step 1: Mean Centering

Each feature is centered: `X_centered = X − μ`

where `μ` is the vector of feature-wise means.

---

### Step 2: Covariance Matrix

The covariance matrix is:

`Σ = (1 / (n − 1)) X_centeredᵀ X_centered`

---

### Step 3: Eigen-Decomposition

Solve:

`Σ v = λ v`

- `v` → principal directions (eigenvectors)
- `λ` → variance explained (eigenvalues)

---

### Step 4: Projection

Project data onto the top `k` eigenvectors:

`Z = X_centered V_k`

This yields a lower-dimensional representation while preserving variance.


## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rice_ml.unsupervised_learning.pca import PCA
from rice_ml.processing.preprocessing import standardize

## Dataset Description

The Wine Quality dataset contains physicochemical properties of red wines.
Each observation includes 11 continuous input variables such as acidity,
chlorides, sulphates, and alcohol content, along with a wine quality score.

Although PCA is unsupervised, this dataset is a good example because:

- the features are numeric
- there are several variables measured on different scales
- many variables are correlated


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")
df.head()

## Exploratory Data Analysis

Before fitting PCA, it is useful to look at the data structure.

In particular, we check:

- feature distributions
- differences in scale
- correlation patterns

Because PCA is driven by variance, preprocessing matters a lot.


### Feature Distributions

Looking at the marginal distribution of each variable helps us see:

- skewness
- possible outliers
- spread across variables
- whether some variables have much larger variation than others

Since PCA is a variance-based method, variables with larger scale can dominate
the components if we do not standardize first.


In [ ]:
df.hist(figsize=(16, 12), bins=20)
plt.suptitle("Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

### Summary Statistics

In [ ]:
summary_stats = df.describe().T[["mean", "std", "min", "max"]]
summary_stats

### Correlation Structure

PCA is especially useful when several features move together.
A correlation matrix helps us see whether some variables may carry overlapping
information.


In [ ]:
corr = df.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Prepare Features

In [ ]:
X = df.values
feature_names = df.columns.tolist()

print("Shape of dataset:", X.shape)
print("Columns:", feature_names)

## Standardization

PCA is sensitive to feature scale, so we standardize the data before fitting
the model. This ensures that each feature contributes on a comparable basis.


In [ ]:
X_std = standardize(X)

print("Mean of first standardized feature:", np.mean(X_std[:, 0]))
print("Std of first standardized feature:", np.std(X_std[:, 0]))

## Fit PCA on All Components

In [ ]:
pca_full = PCA(n_components=X_std.shape[1])
X_pca_full = pca_full.fit_transform(X_std)

print("Explained Variance:", pca_full.explained_variance_)
print("Explained Variance Ratio:", pca_full.explained_variance_ratio_)
print("Total Explained Variance:", pca_full.explained_variance_ratio_.sum())

### Interpretation

Key observations from the explained variance ratios:

- the first few components explain a large share of total variance
- later components contribute much less
- most of the structure may be summarized in relatively few directions

This suggests that the data lies close to a lower-dimensional subspace and that
dimensionality reduction may be appropriate.


## Scree Plot Analysis

A scree plot shows how much variance each principal component explains.
It is commonly used to:

- identify diminishing returns across components
- choose a reasonable cutoff for dimensionality reduction
- visualize the effective dimensionality of the data


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(pca_full.explained_variance_ratio_) + 1),
    pca_full.explained_variance_ratio_,
    marker="o"
)
plt.title("Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.grid(True, alpha=0.3)
plt.show()

## Cumulative Explained Variance

In [ ]:
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(y=0.90, linestyle="--")
plt.axhline(y=0.95, linestyle="--")
plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance Explained")
plt.grid(True, alpha=0.3)
plt.show()

cum_var

### Choosing a Smaller Number of Components

In practice, one common approach is to choose `k` by:

- keeping enough components to explain most of the variance
- locating the elbow in the scree plot
- selecting a small number of components for visualization

Next, we keep only two components so we can visualize the dataset in 2D.


In [ ]:
# Fit PCA with 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

## 2D PCA Projection

To visualize the high-dimensional data, we project each observation onto the
first two principal components:

`Z = X_std W_{1:2}`

where:

- `X_std` is the standardized dataset
- `W_{1:2}` contains the first two principal directions

This gives a 2D view that captures the strongest sources of variation.


In [ ]:
y = df["quality"].values

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    cmap="viridis",
    edgecolor="k",
    s=50
)
plt.colorbar(scatter, label="Wine Quality")
plt.title("2D PCA Projection of Wine Quality Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

## Principal Component Directions

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=["PC1", "PC2"]
)
loadings

## When to Use PCA

PCA is often useful for:

- visualizing high-dimensional data
- reducing noise
- preprocessing before clustering or classification
- addressing multicollinearity

PCA is usually not the best choice when:

- interpretability of original features is essential
- the data is categorical
- the important structure is strongly nonlinear


## Limitations of PCA

Even though PCA is very useful, it has important limitations:

- it only captures linear relationships
- it can be sensitive to outliers
- it maximizes variance, not predictive performance
- individual components may be hard to interpret

Still, PCA remains a foundational tool for visualization, compression, and
preprocessing.


## Conclusion

In this notebook, we applied **Principal Component Analysis** from scratch
using the `rice_ml` package and the Wine Quality dataset.

The main takeaways are:

- PCA finds directions that preserve as much variance as possible
- standardization is important before fitting PCA
- a relatively small number of components can summarize much of the data
- PCA gives a convenient low-dimensional view for exploration and preprocessing

Overall, PCA is a practical and widely used unsupervised learning technique for
understanding structure in multivariate datasets.
